In [15]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / 'src'

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from credit_default.data import load_credit_data
from credit_default.features import clean_credit_data, split_features_target, get_feature_groups

from sklearn.model_selection import train_test_split

from credit_default.data import load_credit_data
from credit_default.features import (
    clean_credit_data,
    split_features_target,
    get_feature_groups
)

from credit_default.modeling.pipelines import build_logistic_regression_pipeline

from credit_default.evaluation import (
    evaluate_classifier,
    threshold_report,
    add_business_utility
)

In [2]:
raw_data = load_credit_data()

In [3]:
raw_data

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000,1,3,1,39,0,0,0,0,...,88004,31237,15980,8500,20000,5003,3047,5000,1000,0
29996,29997,150000,1,3,2,43,-1,-1,-1,-1,...,8979,5190,0,1837,3526,8998,129,0,0,0
29997,29998,30000,1,2,2,37,4,3,2,-1,...,20878,20582,19357,0,0,22000,4200,2000,3100,1
29998,29999,80000,1,3,1,41,1,-1,0,0,...,52774,11855,48944,85900,3409,1178,1926,52964,1804,1


In [4]:
data = clean_credit_data(raw_data)

In [5]:
data.info()

<class 'pandas.DataFrame'>
Index: 29986 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   LIMIT_BAL                   29986 non-null  int64
 1   SEX                         29986 non-null  int64
 2   EDUCATION                   29986 non-null  int64
 3   AGE                         29986 non-null  int64
 4   PAY_0                       29986 non-null  int64
 5   PAY_2                       29986 non-null  int64
 6   PAY_3                       29986 non-null  int64
 7   PAY_4                       29986 non-null  int64
 8   PAY_5                       29986 non-null  int64
 9   PAY_6                       29986 non-null  int64
 10  BILL_AMT1                   29986 non-null  int64
 11  BILL_AMT2                   29986 non-null  int64
 12  BILL_AMT3                   29986 non-null  int64
 13  BILL_AMT4                   29986 non-null  int64
 14  BILL_AMT5             

In [6]:
X, y = split_features_target(data)

In [7]:
X

,LIMIT_BAL,SEX,EDUCATION,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,MARRIAGE_CLEAN
0,20000,2,2,24,2,2,-1,-1,-2,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,26,-1,2,0,0,0,2,...,3272,3455,3261,0,1000,1000,1000,0,2000,2
2,90000,2,2,34,0,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,2
3,50000,2,2,37,0,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,1
4,50000,1,2,57,-1,0,-1,0,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,1,3,39,0,0,0,0,0,0,...,88004,31237,15980,8500,20000,5003,3047,5000,1000,1
29996,150000,1,3,43,-1,-1,-1,-1,0,0,...,8979,5190,0,1837,3526,8998,129,0,0,2
29997,30000,1,2,37,4,3,2,-1,0,0,...,20878,20582,19357,0,0,22000,4200,2000,3100,2
29998,80000,1,3,41,1,-1,0,0,0,-1,...,52774,11855,48944,85900,3409,1178,1926,52964,1804,1


In [8]:
y

0        1
1        1
2        0
3        0
4        0
        ..
29995    0
29996    0
29997    1
29998    1
29999    1
Name: default payment next month, Length: 29986, dtype: int64

In [9]:
feature_groups = get_feature_groups()

In [10]:
feature_groups

FeatureGroups(categorical_cols=['SEX', 'EDUCATION', 'MARRIAGE_CLEAN', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'], numeric_cols=['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'], bill_cols=['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6'], pay_amount_cols=['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'], payment_status_cols=['PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'])

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    stratify = y,
    random_state= 42
)

In [12]:
model = build_logistic_regression_pipeline(
    numeric_cols = feature_groups.numeric_cols,
    categorical_cols = feature_groups.categorical_cols
)

In [13]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [14]:
y_proba = model.predict_proba(X_test)[:, 1]

In [16]:
evaluate_classifier(
    y_true = y_test,
    y_proba = y_proba,
    threshold = 0.5
)

{'threshold': 0.5,
 'accuracy': 0.7394131377125709,
 'precision': 0.4252217997465146,
 'recall': 0.5056518462697814,
 'f1': 0.4619621342512909,
 'roc_auc': 0.7279561055669537,
 'pr_auc': 0.4763169743468314,
 'tn': np.int64(3764),
 'fp': np.int64(907),
 'fn': np.int64(656),
 'tp': np.int64(671)}

In [17]:
threshold_df = threshold_report(
    y_true = y_test,
    y_proba = y_proba
)

In [18]:
threshold_df

,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,0.05,0.223241,0.221591,0.999246,0.362741,0.727956,0.476317,13,4658,1,1326
1,0.10,0.227242,0.222390,0.998493,0.363761,0.727956,0.476317,38,4633,2,1325
2,0.15,0.242081,0.225388,0.995479,0.367557,0.727956,0.476317,131,4540,6,1321
3,0.20,0.264255,0.230056,0.990957,0.373420,0.727956,0.476317,270,4401,12,1315
4,0.25,0.304935,0.238595,0.977393,0.383558,0.727956,0.476317,532,4139,30,1297
5,0.30,0.363621,0.251893,0.952524,0.398424,0.727956,0.476317,917,3754,63,1264
6,0.35,0.445982,0.272562,0.901281,0.418548,0.727956,0.476317,1479,3192,131,1196
7,0.40,0.576025,0.313153,0.767898,0.444881,0.727956,0.476317,2436,2235,308,1019
8,0.45,0.686062,0.372477,0.611907,0.463074,0.727956,0.476317,3303,1368,515,812
9,0.50,0.739413,0.425222,0.505652,0.461962,0.727956,0.476317,3764,907,656,671


In [19]:
utility_df = add_business_utility(threshold_df)

In [20]:
utility_df

,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp,business_utility
11,0.60,0.786762,0.522684,0.416729,0.463732,0.727956,0.476317,4166,505,774,553,-20900
12,0.65,0.793765,0.548180,0.385833,0.452897,0.727956,0.476317,4249,422,815,512,-24800
13,0.70,0.799600,0.576499,0.354936,0.439366,0.727956,0.476317,4325,346,856,471,-30100
10,0.55,0.769423,0.477383,0.445365,0.460819,0.727956,0.476317,4024,647,736,591,-30300
9,0.50,0.739413,0.425222,0.505652,0.461962,0.727956,0.476317,3764,907,656,671,-42300
14,0.75,0.802434,0.611987,0.292389,0.395716,0.727956,0.476317,4425,246,939,388,-51600
8,0.45,0.686062,0.372477,0.611907,0.463074,0.727956,0.476317,3303,1368,515,812,-64000
15,0.80,0.802601,0.647423,0.236624,0.346578,0.727956,0.476317,4500,171,1013,314,-73600
16,0.85,0.800267,0.672000,0.189902,0.296122,0.727956,0.476317,4548,123,1075,252,-95000
17,0.90,0.798600,0.742857,0.137151,0.231552,0.727956,0.476317,4608,63,1145,182,-118000
